In [14]:
!pip install pyspark #install yg diperlukan
!pip install pandas
!pip install matplotlib
!pip install seaborn

In [15]:
# Example: Linear Regression with Spark MLlib
from pyspark.sql import SparkSession
from pyspark.ml.regression import LinearRegression
from pyspark.ml.feature import VectorAssembler

# Initialize Spark Session
spark = SparkSession.builder.appName('MLlib Example').getOrCreate()

# Load sample data
data = [(1, 5.0, 20.0), (2, 10.0, 25.0), (3, 15.0, 30.0), (4, 20.0, 35.0)]
columns = ['ID', 'Feature', 'Target']
df = spark.createDataFrame(data, columns)

# Prepare data for modeling
assembler = VectorAssembler(inputCols=['Feature'], outputCol='Features')
df_transformed = assembler.transform(df)

# Train a linear regression model
lr = LinearRegression(featuresCol='Features', labelCol='Target')
model = lr.fit(df_transformed)

# Print model coefficients
print(f"Coefficients: {model.coefficients}")
print(f"Intercept: {model.intercept}")


25/12/03 23:22:43 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
25/12/03 23:22:44 WARN Instrumentation: [ef9b4bfb] regParam is zero, which might cause numerical instability and overfitting.
                                                                                

Coefficients: [0.9999999999999992]
Intercept: 15.000000000000009


In [16]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.linalg import Vectors
from pyspark.sql import SparkSession

# Spark session
spark = SparkSession.builder.appName("LogReg-Fixed").getOrCreate()

# Dataset dengan DenseVector
data = [
    (1, Vectors.dense([2.0, 3.0]), 0),
    (2, Vectors.dense([1.0, 5.0]), 1),
    (3, Vectors.dense([2.5, 4.5]), 1),
    (4, Vectors.dense([3.0, 6.0]), 0)
]

columns = ["ID", "Features", "Label"]
df = spark.createDataFrame(data, columns)

# Train logistic regression model
lr = LogisticRegression(featuresCol="Features", labelCol="Label")
model = lr.fit(df)

# Print coefficients & intercept
print("Coefficients:", model.coefficients)
print("Intercept:", model.intercept)


25/12/03 23:22:49 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Coefficients: [-12.262057917018774,4.0873522624320255]
Intercept: 11.568912714495273


In [17]:
from pyspark.sql import SparkSession
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import VectorAssembler

spark = SparkSession.builder.appName("KMeans").getOrCreate()

# Dataset awal
data = [
    (1, 1.0, 1.0),
    (2, 5.0, 5.0),
    (3, 10.0, 10.0),
    (4, 15.0, 15.0)
]

df = spark.createDataFrame(data, ["ID", "x", "y"])

# Ubah menjadi FEATURES VECTOR
assembler = VectorAssembler(inputCols=["x", "y"], outputCol="Features")
df_vec = assembler.transform(df)

# KMeans
kmeans = KMeans(featuresCol="Features", k=2)
model = kmeans.fit(df_vec)

# Hasil
print("Cluster Centers:", model.clusterCenters())



25/12/03 23:22:54 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Cluster Centers: [array([12.5, 12.5]), array([3., 3.])]


In [18]:
#HOMEWORK
#1. Load a real-world dataset into Spark
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler

spark = SparkSession.builder.appName("Homework-Diabetes-Classification").getOrCreate()

df = spark.read.csv("diabetes.csv", header=True, inferSchema=True)

print("Schema Dataset:")
df.printSchema()
print("Jumlah Baris:", df.count())


25/12/03 23:23:00 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Schema Dataset:
root
 |-- Pregnancies: integer (nullable = true)
 |-- Glucose: integer (nullable = true)
 |-- BloodPressure: integer (nullable = true)
 |-- SkinThickness: integer (nullable = true)
 |-- Insulin: integer (nullable = true)
 |-- BMI: double (nullable = true)
 |-- DiabetesPedigreeFunction: double (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Outcome: integer (nullable = true)

Jumlah Baris: 768


In [19]:
#2. Build a classification model using Spark MLlib
from pyspark.ml.classification import LogisticRegression

# daftar fitur (semua kolom kecuali Outcome)
feature_cols = [col for col in df.columns if col != "Outcome"]

# ubah fitur menjadi vector
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_vec = assembler.transform(df)

# scaling
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures",
                        withMean=True, withStd=True)
df_scaled = scaler.fit(df_vec).transform(df_vec)

# split data
train_df, test_df = df_scaled.randomSplit([0.8, 0.2], seed=42)

# model klasifikasi
lr = LogisticRegression(featuresCol="scaledFeatures", labelCol="Outcome")
model = lr.fit(train_df)

# prediksi awal
predictions = model.transform(test_df)


In [20]:
#2. Evaluate its performance
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(labelCol="Outcome",
                                          rawPredictionCol="rawPrediction")

auc = evaluator.evaluate(predictions)
print("AUC awal:", auc)


AUC awal: 0.8619186046511628


In [21]:
#3. Hyperparameter tuning using cross-validation
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# grid parameter untuk dicoba
paramGrid = (ParamGridBuilder()
             .addGrid(lr.regParam, [0.0, 0.01, 0.1, 0.5])
             .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])
             .build())

# cross-validator
cv = CrossValidator(estimator=lr,
                    estimatorParamMaps=paramGrid,
                    evaluator=evaluator,
                    numFolds=5)

# latih model terbaik
cv_model = cv.fit(train_df)
best_model = cv_model.bestModel

# evaluasi ulang dengan model terbaik
best_predictions = best_model.transform(test_df)
best_auc = evaluator.evaluate(best_predictions)

print("AUC setelah tuning:", best_auc)

AUC setelah tuning: 0.8619186046511627
